# Graficas de medidas con delay

Estas medidas comparan el RTT y la perdida de paquetes obtenidos con `ping` hacia M2 y M3 usando paquetes de 120 bytes. Se contrastan las pruebas sin delay con las pruebas donde `tc netem` aplica retardos de 100 ms, 200 ms y 500 ms. Las graficas generadas se guardan en esta misma carpeta.

In [ ]:
from pathlib import Path
import html
import re
from statistics import mean

CARPETA = Path.cwd()


def delay_configurado(path):
    if 'sin_delay' in path.name:
        return 0
    return int(re.search(r'delay_(\d+)ms', path.name).group(1))


def parse_medida(path):
    texto = path.read_text(encoding='utf-8', errors='replace')
    delay_ms = delay_configurado(path)
    destino = None
    muestras = {}
    perdida = {}

    for linea in texto.splitlines():
        cabecera = re.match(r'Ping\s+\d+\s+bytes\s+(M\d+)', linea)
        if cabecera:
            destino = cabecera.group(1)
            muestras.setdefault(destino, [])
            continue

        if destino:
            rtt = re.search(r'time=([0-9.]+)\s*ms', linea)
            if rtt:
                muestras[destino].append(float(rtt.group(1)))
                continue

            loss = re.search(r'(\d+(?:\.\d+)?)% packet loss', linea)
            if loss:
                perdida[destino] = float(loss.group(1))

    filas = []
    for host in sorted(set(muestras) | set(perdida)):
        valores = muestras.get(host, [])
        filas.append({
            'archivo': path.name,
            'delay_ms': delay_ms,
            'destino': host,
            'n_recibidos': len(valores),
            'rtt_medio_ms': mean(valores) if valores else None,
            'rtt_min_ms': min(valores) if valores else None,
            'rtt_max_ms': max(valores) if valores else None,
            'perdida_pct': perdida.get(host, 100.0 if not valores else 0.0),
            'muestras': valores,
        })
    return filas


datos = []
for archivo in sorted(CARPETA.glob('*.txt')):
    datos.extend(parse_medida(archivo))

for fila in sorted(datos, key=lambda x: (x['destino'], x['delay_ms'])):
    rtt = 'sin respuesta' if fila['rtt_medio_ms'] is None else f"{fila['rtt_medio_ms']:.3f} ms"
    print(f"delay={fila['delay_ms']:3} ms {fila['destino']}: RTT medio={rtt}, perdida={fila['perdida_pct']:.0f}%")


def color(destino):
    return '#2878b5' if destino == 'M2' else '#d95f02'


def svg_lineas(nombre, titulo, campo, etiqueta_y, maximo=None):
    filas = sorted(datos, key=lambda x: (x['destino'], x['delay_ms']))
    delays = sorted({f['delay_ms'] for f in filas})
    destinos = sorted({f['destino'] for f in filas})
    valores = [f[campo] for f in filas if f[campo] is not None]
    if not valores:
        return
    ymax = maximo if maximo is not None else max(valores) * 1.12
    ymax = ymax or 1

    ancho = 900
    alto = 520
    margen_izq = 75
    margen_der = 45
    margen_sup = 60
    margen_inf = 75
    base = alto - margen_inf
    area_w = ancho - margen_izq - margen_der
    area_h = alto - margen_sup - margen_inf
    xmin, xmax = min(delays), max(delays)
    xrango = xmax - xmin or 1

    def x_pos(delay):
        return margen_izq + ((delay - xmin) / xrango) * area_w

    def y_pos(valor):
        return base - (valor / ymax) * area_h

    partes = [
        f"<svg xmlns='http://www.w3.org/2000/svg' width='{ancho}' height='{alto}' viewBox='0 0 {ancho} {alto}'>",
        "<rect width='100%' height='100%' fill='white'/>",
        f"<text x='{ancho/2}' y='30' text-anchor='middle' font-family='Arial' font-size='22' font-weight='700'>{html.escape(titulo)}</text>",
        f"<line x1='{margen_izq}' y1='{base}' x2='{ancho-margen_der}' y2='{base}' stroke='#333'/>",
        f"<line x1='{margen_izq}' y1='{margen_sup}' x2='{margen_izq}' y2='{base}' stroke='#333'/>",
        f"<text x='{ancho/2}' y='{alto-20}' text-anchor='middle' font-family='Arial' font-size='14'>Delay configurado (ms)</text>",
        f"<text x='22' y='{margen_sup + area_h/2}' transform='rotate(-90 22 {margen_sup + area_h/2})' text-anchor='middle' font-family='Arial' font-size='14'>{html.escape(etiqueta_y)}</text>",
    ]

    for i in range(6):
        v = ymax * i / 5
        y = y_pos(v)
        partes.append(f"<line x1='{margen_izq}' y1='{y:.1f}' x2='{ancho-margen_der}' y2='{y:.1f}' stroke='#ddd'/>")
        partes.append(f"<text x='{margen_izq-10}' y='{y+4:.1f}' text-anchor='end' font-family='Arial' font-size='12'>{v:.1f}</text>")

    for delay in delays:
        x = x_pos(delay)
        partes.append(f"<line x1='{x:.1f}' y1='{base}' x2='{x:.1f}' y2='{base+6}' stroke='#333'/>")
        partes.append(f"<text x='{x:.1f}' y='{base+24}' text-anchor='middle' font-family='Arial' font-size='12'>{delay}</text>")

    offset_destino = {'M2': -8, 'M3': 8}
    for destino in destinos:
        puntos = []
        for fila in [f for f in filas if f['destino'] == destino]:
            valor = fila[campo]
            if valor is not None:
                puntos.append((x_pos(fila['delay_ms']) + offset_destino.get(destino, 0), y_pos(valor), valor))
        path = ' '.join(('M' if i == 0 else 'L') + f" {x:.1f} {y:.1f}" for i, (x, y, _) in enumerate(puntos))
        partes.append(f"<path d='{path}' fill='none' stroke='{color(destino)}' stroke-width='2.5'/>")
        for x, y, valor in puntos:
            partes.append(f"<circle cx='{x:.1f}' cy='{y:.1f}' r='4.5' fill='{color(destino)}'/>")
            partes.append(f"<text x='{x:.1f}' y='{y-9:.1f}' text-anchor='middle' font-family='Arial' font-size='11'>{valor:.1f}</text>")

    partes.append("<circle cx='720' cy='56' r='5' fill='#2878b5'/><text x='735' y='61' font-family='Arial' font-size='13'>M2</text>")
    partes.append("<circle cx='720' cy='80' r='5' fill='#d95f02'/><text x='735' y='85' font-family='Arial' font-size='13'>M3</text>")
    partes.append('</svg>')
    (CARPETA / nombre).write_text('\n'.join(partes), encoding='utf-8')


svg_lineas('rtt_medio_delay_120bytes.svg', 'RTT medio frente al delay configurado', 'rtt_medio_ms', 'RTT medio (ms)')
svg_lineas('perdida_paquetes_delay_120bytes.svg', 'Perdida de paquetes frente al delay configurado', 'perdida_pct', 'Perdida (%)', maximo=100)

print('\nGraficas guardadas:')
print(CARPETA / 'rtt_medio_delay_120bytes.svg')
print(CARPETA / 'perdida_paquetes_delay_120bytes.svg')